###Installing Required Packages

In [0]:
# %pip install openmeteo-requests pvlib pandas pyarrow xgboost scikit-learn seaborn
# dbutils.library.restartPython()

### Testing API call 
with 1-Month of historic data

In [0]:
# import openmeteo_requests
# import pandas as pd

# # 1. Fetch historical data (Perth, WA)
# om = openmeteo_requests.Client()
# params = {
# 	"latitude": -31.95,
# 	"longitude": 115.86,
# 	"start_date": "2024-01-01",
# 	"end_date": "2024-01-31",
# 	"hourly": ["direct_normal_irradiance", "diffuse_radiation"]
# }
# response = om.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)[0]
# hourly = response.Hourly()

# # 2. Format into a Pandas DataFrame
# date_range = pd.date_range(
#     start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
#     end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
#     freq=pd.Timedelta(seconds=hourly.Interval()),
#     inclusive="left"
# )

# pdf_raw = pd.DataFrame({
#     "timestamp": date_range,
#     "dni": hourly.Variables(0).ValuesAsNumpy(),
#     "dhi": hourly.Variables(1).ValuesAsNumpy()
# })

# # 3. Convert to a Distributed Spark DataFrame (Bronze Layer)
# df_bronze = spark.createDataFrame(pdf_raw)
# df_bronze.display()

### Testing feature engineering
applied geometric effect correction

In [0]:
# import pvlib
# import numpy as np
# from pyspark.sql.functions import pandas_udf
# import pyspark.sql.types as T
# import pandas as pd

# # Define panel geometry
# TILT = 32.0 
# AZIMUTH = 0.0 # Facing North
# LAT = -31.95
# LON = 115.86

# @pandas_udf("float")
# def calculate_gti(timestamps: pd.Series, dni: pd.Series, dhi: pd.Series) -> pd.Series:
#     # 1. Calculate Sun Position (Convert to DatetimeIndex for pvlib)
#     solpos = pvlib.solarposition.get_solarposition(pd.DatetimeIndex(timestamps), LAT, LON)
    
#     # Strip the DatetimeIndex to prevent Pandas outer-join alignment errors
#     zenith = solpos['apparent_zenith'].to_numpy()
#     azimuth = solpos['azimuth'].to_numpy()
    
#     # 2. Calculate Global Tilted Irradiance (GTI)
#     poa_irrad = pvlib.irradiance.get_total_irradiance(
#         surface_tilt=TILT,
#         surface_azimuth=AZIMUTH,
#         dni=dni,
#         ghi=dhi + (dni * np.cos(np.radians(zenith))), 
#         dhi=dhi,
#         solar_zenith=zenith,
#         solar_azimuth=azimuth
#     )
    
#     return poa_irrad['poa_global']

# # 3. Apply function
# df_silver = df_bronze.withColumn(
#     "global_tilted_irradiance[W/m^2]", 
#     calculate_gti(df_bronze["timestamp"], df_bronze["dni"], df_bronze["dhi"])
# )

# df_silver.display()

### Data Ingestion
Collect variables over 10-year window with yearly batching to avoid memory buffer overflows

In [0]:
import openmeteo_requests
import pandas as pd

om = openmeteo_requests.Client()
years = list(range(2016, 2026))
all_dfs = []

print("Starting cloud-tier batched data ingestion...")
for year in years:
    print(f"Fetching data for year: {year}...")
    
    params = {
        "latitude": -31.95,
        "longitude": 115.86,
        "start_date": f"{year}-01-01",
        "end_date": f"{year}-12-31",
        "hourly": [
            "diffuse_radiation",
            "cloud_cover_low",   
            "cloud_cover_mid",   
            "cloud_cover_high",  
            "total_column_integrated_water_vapour",
            "sunshine_duration",
            "direct_normal_irradiance",
            "temperature_2m",        
            "relative_humidity_2m",  
            "surface_pressure"       
        ]
    }
    
    response = om.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)[0]
    hourly = response.Hourly()
    
    date_range = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )
    
    pdf_year = pd.DataFrame({
        "timestamp": date_range,
        "dhi": hourly.Variables(0).ValuesAsNumpy(),
        "cloud_low": hourly.Variables(1).ValuesAsNumpy(),   
        "cloud_mid": hourly.Variables(2).ValuesAsNumpy(),   
        "cloud_high": hourly.Variables(3).ValuesAsNumpy(),  
        "water_vapour": hourly.Variables(4).ValuesAsNumpy(),
        "sunshine_duration": hourly.Variables(5).ValuesAsNumpy(),
        "dni": hourly.Variables(6).ValuesAsNumpy(),
        "temperature": hourly.Variables(7).ValuesAsNumpy(),      
        "relative_humidity": hourly.Variables(8).ValuesAsNumpy(),
        "surface_pressure": hourly.Variables(9).ValuesAsNumpy()   
    })
    
    all_dfs.append(pdf_year)

pdf_combined = pd.concat(all_dfs, ignore_index=True)
df_bronze = spark.createDataFrame(pdf_combined)
print(f"Ingestion complete! Total rows loaded: {df_bronze.count()}")

In [0]:
import pvlib
import pandas as pd
import numpy as np
from pyspark.sql.functions import pandas_udf, col, when
import pyspark.sql.types as T

LAT = -31.95
LON = 115.86

# 1. Expand the UDF schema to return Zenith along with theoretical DNI/GHI
cs_schema = T.StructType([
    T.StructField("clear_dni", T.FloatType(), True),
    T.StructField("clear_ghi", T.FloatType(), True),
    T.StructField("zenith", T.FloatType(), True)  
])

@pandas_udf(cs_schema)
def get_clear_sky_and_zenith(timestamps: pd.Series) -> pd.DataFrame:
    tus = pd.DatetimeIndex(timestamps)
    loc = pvlib.location.Location(LAT, LON)
    
    # Get clear sky components
    cs = loc.get_clearsky(tus)
    # Get solar position matrix to extract zenith
    solpos = loc.get_solarposition(tus)
    
    return pd.DataFrame({
        "clear_dni": cs['dni'].astype('float32'),
        "clear_ghi": cs['ghi'].astype('float32'),
        "zenith": solpos['apparent_zenith'].astype('float32')
    })

# 2. Extract components into the Spark DataFrame
df_theoretical = df_bronze.withColumn(
    "cs_components", get_clear_sky_and_zenith(col("timestamp"))
).select(
    "*",
    col("cs_components.clear_dni").alias("cs_dni"),
    col("cs_components.clear_ghi").alias("cs_ghi"),
    col("cs_components.zenith").alias("zenith")
).drop("cs_components")

# Calculate Factors and normalize tiered cloud features
df_silver_factors = df_theoretical.withColumn(
    "direct_clear_sky_factor",
    when((col("cs_dni") > 5.0) & (col("zenith") < 85.0), col("dni") / col("cs_dni")).otherwise(0.0)
).withColumn(
    "diffuse_clear_sky_factor",
    when((col("cs_ghi") > 5.0) & (col("zenith") < 85.0), col("dhi") / col("cs_ghi")).otherwise(0.0)
).withColumn(
    "direct_clear_sky_factor",
    when(col("direct_clear_sky_factor") > 1.0, 1.0).otherwise(col("direct_clear_sky_factor"))
).withColumn(
    "diffuse_clear_sky_factor",
    when(col("diffuse_clear_sky_factor") > 1.0, 1.0).otherwise(col("diffuse_clear_sky_factor"))
).withColumn(
    "sunshine_fraction", col("sunshine_duration") / 3600.0
).withColumn(
    "cloud_low_fraction", col("cloud_low") / 100.0   
).withColumn(
    "cloud_mid_fraction", col("cloud_mid") / 100.0   
).withColumn(
    "cloud_high_fraction", col("cloud_high") / 100.0 
).withColumn(
    "rh_fraction", col("relative_humidity") / 100.0  
).select(
    "timestamp",
    "cloud_low_fraction",
    "cloud_mid_fraction",
    "cloud_high_fraction",
    "water_vapour",
    "sunshine_fraction",
    "temperature",        
    "rh_fraction",         
    "surface_pressure",    
    "direct_clear_sky_factor",
    "diffuse_clear_sky_factor"
)

print("Layered Cloud Silver Layer complete!")
# df_silver_factors.display()

### Hyperparameter tunning

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# 1. Prepare Data with updated feature column names
data = df_silver_factors.select(
    "cloud_low_fraction", "cloud_mid_fraction", "cloud_high_fraction", 
    "water_vapour", "sunshine_fraction", 
    "temperature", "rh_fraction", "surface_pressure",
    "direct_clear_sky_factor", "diffuse_clear_sky_factor"
).toPandas()

# Updated feature array
X = data[["cloud_low_fraction", "cloud_mid_fraction", "cloud_high_fraction", "water_vapour", "sunshine_fraction", "temperature", "rh_fraction", "surface_pressure"]]
y = data[["direct_clear_sky_factor", "diffuse_clear_sky_factor"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

depths = range(3, 22, 2)
train_errors = []
test_errors = []

for depth in depths:
    base_model = XGBRegressor(n_estimators=500, max_depth=depth, learning_rate=0.01, reg_alpha=0.1, reg_lambda=1.0, random_state=42, subsample=0.6, n_jobs=-1, objective="reg:logistic")
    model_suite = MultiOutputRegressor(base_model)
    model_suite.fit(X_train, y_train)
    
    train_errors.append(mean_squared_error(y_train, model_suite.predict(X_train)))
    test_errors.append(mean_squared_error(y_test, model_suite.predict(X_test)))
    print(f"Depth {depth:02d} -> Train MSE: {train_errors[-1]:.5f} | Test MSE: {test_errors[-1]:.5f}")


# 3. Plot Complexity Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(depths, train_errors, 'o-', color='gray', label='Training Loss (Bias)')
plt.plot(depths, test_errors, 'o-', color='crimson', lw=2, label='Testing Loss (Variance)')
plt.title("XGBoost Complexity Elbow Plot (Model Depth Tuning)")
plt.xlabel("Model Complexity (max_depth)")
plt.ylabel("Total Mean Squared Error")
plt.xticks(depths)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

###Final Training & Model Registration

In [0]:
import mlflow
import mlflow.sklearn
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

data = df_silver_factors.select(
    "cloud_low_fraction", "cloud_mid_fraction", "cloud_high_fraction", 
     "water_vapour", "sunshine_fraction", 
    "temperature", "rh_fraction", "surface_pressure",
    "direct_clear_sky_factor", "diffuse_clear_sky_factor"
).toPandas()

X = data[["cloud_low_fraction", "cloud_mid_fraction", "cloud_high_fraction",  "water_vapour", "sunshine_fraction", "temperature", "rh_fraction", "surface_pressure"]]
y = data[["direct_clear_sky_factor", "diffuse_clear_sky_factor"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Train using your tuned hyperparameters (optimal depth, row subsampling, and L1/L2 penalties)
with mlflow.start_run(run_name="Production_Solar_Model_v2") as run:
    
    base_model = XGBRegressor(
        n_estimators=500,        # Match your successful hyperparameter tuning loop
        learning_rate=0.01,      # Smooth learning rate to capture subtle nuances
        max_depth=10,             # Sweets-pot depth from your 10-feature elbow scan
        subsample=0.6,           # Introduces row variation to avoid local noise memorization
        reg_alpha=0.1,           # Regularizes the expanded feature space
        reg_lambda=1.0,
        n_jobs=-1, 
        objective="reg:logistic",          
        random_state=42
    )
    
    prod_model = MultiOutputRegressor(base_model)
    prod_model.fit(X_train, y_train)
    
    # 3. Log performance metrics
    preds = prod_model.predict(X_test)
    mse_direct = mean_squared_error(y_test.iloc[:, 0], preds[:, 0])
    mse_diffuse = mean_squared_error(y_test.iloc[:, 1], preds[:, 1])
    
    mlflow.log_metric("optimal_mse_direct", mse_direct)
    mlflow.log_metric("optimal_mse_diffuse", mse_diffuse)
    
    # 4. Register the model asset to MLflow
    mlflow.sklearn.log_model(
        sk_model=prod_model, 
        name="solar_factor_model",
        input_example=X_train.head(3)
    )
    
    print("-" * 60)
    print("OPTIMIZED PRODUCTION MODEL REGISTERED!")
    print(f"Copy this Run ID for the inference notebook: {run.info.run_id}")
    print("-" * 60)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Generate test set predictions
predictions = prod_model.predict(X_test)

# 2. Initialize a 2x2 diagnostic plotting matrix
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# --- DIRECT CLEAR SKY FACTOR DIAGNOSTICS ---
# Predicted vs Actual
axs[0, 0].scatter(y_test.iloc[:, 0], predictions[:, 0], alpha=0.1, color='royalblue', s=5)
axs[0, 0].plot([0, 1.05], [0, 1.05], 'r--', lw=2, label="Perfect 1:1 Alignment")
axs[0, 0].set_title("Direct Factor: Predicted vs. Actual")
axs[0, 0].set_xlabel("Actual Factor")
axs[0, 0].set_ylabel("Predicted Factor")
axs[0, 0].legend()

# Residual Distribution
residuals_direct = y_test.iloc[:, 0] - predictions[:, 0]
sns.histplot(residuals_direct, kde=True, ax=axs[0, 1], color='royalblue', bins=50)
axs[0, 1].axvline(0, color='red', linestyle='--', lw=1.5)
axs[0, 1].set_title("Direct Factor: Residual Distribution")
axs[0, 1].set_xlabel("Residual (Actual - Predicted)")

# --- DIFFUSE CLEAR SKY FACTOR DIAGNOSTICS ---
# Predicted vs Actual
axs[1, 0].scatter(y_test.iloc[:, 1], predictions[:, 1], alpha=0.1, color='darkorange', s=5)
axs[1, 0].plot([0, 1.05], [0, 1.05], 'r--', lw=2, label="Perfect 1:1 Alignment")
axs[1, 0].set_title("Diffuse Factor: Predicted vs. Actual")
axs[1, 0].set_xlabel("Actual Factor")
axs[1, 0].set_ylabel("Predicted Factor")
axs[1, 0].legend()

# Residual Distribution
residuals_diffuse = y_test.iloc[:, 1] - predictions[:, 1]
sns.histplot(residuals_diffuse, kde=True, ax=axs[1, 1], color='darkorange', bins=50)
axs[1, 1].axvline(0, color='red', linestyle='--', lw=1.5)
axs[1, 1].set_title("Diffuse Factor: Residual Distribution")
axs[1, 1].set_xlabel("Residual (Actual - Predicted)")

plt.tight_layout()
plt.show()

### Clipping Nighttime Hours

In [0]:
# Select all features and targets from the Silver layer
data = df_silver_factors.select(
    "cloud_low_fraction", "cloud_mid_fraction", "cloud_high_fraction", 
    "water_vapour", "sunshine_fraction", 
    "temperature", "rh_fraction", "surface_pressure",
    "direct_clear_sky_factor", "diffuse_clear_sky_factor"
).toPandas()

# Filter out night hours where sunshine fraction is 0
daytime_data = data[data["sunshine_fraction"] > 0.0].copy()

X_day = daytime_data[["cloud_low_fraction", "cloud_mid_fraction", "cloud_high_fraction", "water_vapour", "sunshine_fraction", "temperature", "rh_fraction", "surface_pressure"]]
y_day = daytime_data[["direct_clear_sky_factor", "diffuse_clear_sky_factor"]]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_day, y_day, test_size=0.2, random_state=42)

print(f"Night hours clipped! Training set rows optimized from {len(data)} down to {len(daytime_data)}")

In [0]:
import mlflow
import mlflow.sklearn
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error

with mlflow.start_run(run_name="Daytime_Optimized_Solar_Model") as run:
    
    base_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.01,
        max_depth=10,
        subsample=0.6,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        objective="reg:logistic"
    )
    
    day_model = MultiOutputRegressor(base_model)
    day_model.fit(X_train, y_train)
    
    # Evaluate performance on the daytime test partition
    preds = day_model.predict(X_test)
    mse_direct = mean_squared_error(y_test.iloc[:, 0], preds[:, 0])
    mse_diffuse = mean_squared_error(y_test.iloc[:, 1], preds[:, 1])
    
    mlflow.log_metric("daytime_mse_direct", mse_direct)
    mlflow.log_metric("daytime_mse_diffuse", mse_diffuse)
    
    mlflow.sklearn.log_model(
        sk_model=day_model, 
        name="solar_factor_model",
        input_example=X_train.head(3)
    )
    
    print("-" * 60)
    print("DAYTIME OPTIMIZED PRODUCTION MODEL REGISTERED!")
    print(f"Copy this Run ID for the inference notebook: {run.info.run_id}")
    print("-" * 60)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# Direct Factor
axs[0, 0].scatter(y_test.iloc[:, 0], preds[:, 0], alpha=0.1, color='royalblue', s=5)
axs[0, 0].plot([0, 1], [0, 1], 'r--', lw=2, label="Perfect 1:1 Alignment")
axs[0, 0].set_title("Daytime Direct Factor: Predicted vs. Actual")
axs[0, 0].set_xlabel("Actual Factor")
axs[0, 0].set_ylabel("Predicted Factor")
axs[0, 0].legend()

residuals_direct = y_test.iloc[:, 0] - preds[:, 0]
sns.histplot(residuals_direct, kde=True, ax=axs[0, 1], color='royalblue', bins=50)
axs[0, 1].axvline(0, color='red', linestyle='--', lw=1.5)
axs[0, 1].set_title("Daytime Direct Factor: Residual Distribution")

# Diffuse Factor
axs[1, 0].scatter(y_test.iloc[:, 1], preds[:, 1], alpha=0.1, color='darkorange', s=5)
axs[1, 0].plot([0, 1], [0, 1], 'r--', lw=2, label="Perfect 1:1 Alignment")
axs[1, 0].set_title("Daytime Diffuse Factor: Predicted vs. Actual")
axs[1, 0].set_xlabel("Actual Factor")
axs[1, 0].set_ylabel("Predicted Factor")
axs[1, 0].legend()

residuals_diffuse = y_test.iloc[:, 1] - preds[:, 1]
sns.histplot(residuals_diffuse, kde=True, ax=axs[1, 1], color='darkorange', bins=50)
axs[1, 1].axvline(0, color='red', linestyle='--', lw=1.5)
axs[1, 1].set_title("Daytime Diffuse Factor: Residual Distribution")

plt.tight_layout()
plt.show()

In [0]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

# 1. Create a temporary DataFrame of your test set actuals and predictions
eval_df = pd.DataFrame({
    "actual_direct": y_test.iloc[:, 0],
    "pred_direct": preds[:, 0],
    "actual_diffuse": y_test.iloc[:, 1],
    "pred_diffuse": preds[:, 1]
})

# 2. Define masks to isolate the continuous interior (excluding the exact 0 and 1 edges)
interior_direct = eval_df[(eval_df["actual_direct"] > 0.01) & (eval_df["actual_direct"] < 0.99)]
interior_diffuse = eval_df[(eval_df["actual_diffuse"] > 0.01) & (eval_df["actual_diffuse"] < 0.99)]

# 3. Calculate scores on the continuous interior slices
mse_dir_int = mean_squared_error(interior_direct["actual_direct"], interior_direct["pred_direct"])
r2_dir_int = r2_score(interior_direct["actual_direct"], interior_direct["pred_direct"])

mse_dif_int = mean_squared_error(interior_diffuse["actual_diffuse"], interior_diffuse["pred_diffuse"])
r2_dif_int = r2_score(interior_diffuse["actual_diffuse"], interior_diffuse["pred_diffuse"])

print("=" * 50)
print(f"INTERIOR PERFORMANCE METRICS (Excluding 0.0 and 1.0 Edges)")
print("=" * 50)
print(f"Direct Factor  -> Interior MSE: {mse_dir_int:.5f} | Interior R²: {r2_dir_int:.3f}")
print(f"Diffuse Factor -> Interior MSE: {mse_dif_int:.5f} | Interior R²: {r2_dif_int:.3f}")
print(f"Evaluated on {len(interior_direct)} direct rows and {len(interior_diffuse)} diffuse rows.")
print("=" * 50)

### Increase the weight of interior data

In [0]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Calculate a sample weight for each row in the training set
# Points in the interior (away from 0 and 1) get a weight of 5.0, edge points get 1.0
def calculate_interior_weights(y_df):
    # Check if either target is sitting on a boundary edge
    is_edge_direct = (y_df.iloc[:, 0] <= 0.01) | (y_df.iloc[:, 0] >= 0.99)
    is_edge_diffuse = (y_df.iloc[:, 1] <= 0.01) | (y_df.iloc[:, 1] >= 0.99)
    
    # If BOTH are in the interior, give it high priority (5x weight)
    weights = np.where(~is_edge_direct & ~is_edge_diffuse, 5.0, 1.0)
    return weights

sample_weights_train = calculate_interior_weights(y_train)

# 2. Retrain inside MLflow using sample weights
with mlflow.start_run(run_name="Weighted_Interior_Solar_Model") as run:
    
    base_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.01,
        max_depth=10,
        subsample=0.6,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        objective="reg:logistic"
    )
    
    weighted_model = MultiOutputRegressor(base_model)
    
    # Pass sample weights downstream through the MultiOutput wrapper to XGBoost
    weighted_model.fit(X_train, y_train, sample_weight=sample_weights_train)
    
    # 3. Evaluate the new predictions
    preds_weighted = weighted_model.predict(X_test)
    
    # Track metrics in MLflow
    mlflow.log_metric("weighted_test_mse_direct", mean_squared_error(y_test.iloc[:, 0], preds_weighted[:, 0]))
    mlflow.log_metric("weighted_test_mse_diffuse", mean_squared_error(y_test.iloc[:, 1], preds_weighted[:, 1]))
    
    mlflow.sklearn.log_model(
        sk_model=weighted_model, 
        name="solar_factor_model",
        input_example=X_train.head(3)
    )
    
    print("-" * 60)
    print("WEIGHTED PRODUCTION MODEL REGISTERED!")
    print(f"New Run ID: {run.info.run_id}")
    print("-" * 60)

In [0]:
eval_weighted_df = pd.DataFrame({
    "actual_direct": y_test.iloc[:, 0],
    "pred_direct": preds_weighted[:, 0],
    "actual_diffuse": y_test.iloc[:, 1],
    "pred_diffuse": preds_weighted[:, 1]
})

interior_dir_w = eval_weighted_df[(eval_weighted_df["actual_direct"] > 0.01) & (eval_weighted_df["actual_direct"] < 0.99)]
interior_dif_w = eval_weighted_df[(eval_weighted_df["actual_diffuse"] > 0.01) & (eval_weighted_df["actual_diffuse"] < 0.99)]

r2_dir_weighted = r2_score(interior_dir_w["actual_direct"], interior_dir_w["pred_direct"])
r2_dif_weighted = r2_score(interior_dif_w["actual_diffuse"], interior_dif_w["pred_diffuse"])

print("=" * 50)
print(f"NEW WEIGHTED INTERIOR R² SCORES")
print("=" * 50)
print(f"Direct Factor  -> Interior R²: {r2_dir_weighted:.3f} (Was 0.737)")
print(f"Diffuse Factor -> Interior R²: {r2_dif_weighted:.3f} (Was 0.241)")
print("=" * 50)

### The Regressor Chain Strategy

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.multioutput import RegressorChain
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# 1. Prepare Data and cleanly drop NaNs before the split
data = df_silver_factors.select(
    "cloud_low_fraction", "cloud_mid_fraction", "cloud_high_fraction", 
    "water_vapour", "sunshine_fraction", "temperature", 
    "rh_fraction", "surface_pressure",
    "direct_clear_sky_factor", "diffuse_clear_sky_factor"
).toPandas()

# Filter out night hours and drop incomplete rows
daytime_data = data[data["sunshine_fraction"] > 0.0].dropna().copy()

X_day = daytime_data[[
    "cloud_low_fraction", "cloud_mid_fraction", "cloud_high_fraction", 
    "water_vapour", "sunshine_fraction", "temperature", 
    "rh_fraction", "surface_pressure"
]]
y_day = daytime_data[["direct_clear_sky_factor", "diffuse_clear_sky_factor"]]

X_train, X_test, y_train, y_test = train_test_split(X_day, y_day, test_size=0.2, random_state=42)

# 2. Compute Sample Weights (5x priority to fluid interior points)
def calculate_interior_weights(y_df):
    is_edge_direct = (y_df.iloc[:, 0] <= 0.01) | (y_df.iloc[:, 0] >= 0.99)
    is_edge_diffuse = (y_df.iloc[:, 1] <= 0.01) | (y_df.iloc[:, 1] >= 0.99)
    return np.where(~is_edge_direct & ~is_edge_diffuse, 5.0, 1.0)

sample_weights_train = calculate_interior_weights(y_train)

# 3. Initialize Base Regressor
base_xgboost = XGBRegressor(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.6,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    objective="reg:logistic"
)

# 4. Wrap in a Regressor Chain [Direct -> Diffuse]
weighted_chain_model = RegressorChain(base_xgboost, order=[0, 1])

with mlflow.start_run(run_name="Weighted_Chained_Solar_Model") as run:
    
    # CRITICAL SYNTAX: Pass sample weights to individual chain estimators via fit_params
    weighted_chain_model.fit(X_train, y_train, sample_weight=sample_weights_train)
    
    # 5. Evaluate the results on the test partition
    preds_weighted_chain = weighted_chain_model.predict(X_test)
    
    eval_df = pd.DataFrame({
        "actual_direct": y_test.iloc[:, 0],
        "pred_direct": preds_weighted_chain[:, 0],
        "actual_diffuse": y_test.iloc[:, 1],
        "pred_diffuse": preds_weighted_chain[:, 1]
    })
    
    # Isolate interior slices for true physical evaluation
    int_dir = eval_df[(eval_df["actual_direct"] > 0.01) & (eval_df["actual_direct"] < 0.99)]
    int_dif = eval_df[(eval_df["actual_diffuse"] > 0.01) & (eval_df["actual_diffuse"] < 0.99)]
    
    r2_dir = r2_score(int_dir["actual_direct"], int_dir["pred_direct"])
    r2_dif = r2_score(int_dif["actual_diffuse"], int_dif["pred_diffuse"])
    
    # Log parameters and artifacts to MLflow
    mlflow.log_metric("weighted_chain_interior_r2_direct", r2_dir)
    mlflow.log_metric("weighted_chain_interior_r2_diffuse", r2_dif)
    mlflow.sklearn.log_model(sk_model=weighted_chain_model, name="solar_factor_model")
    
    print("=" * 50)
    print(f"WEIGHTED CHAINED INTERIOR R² SCORES")
    print("=" * 50)
    print(f"Direct Factor  -> Interior R²: {r2_dir:.3f}")
    print(f"Diffuse Factor -> Interior R²: {r2_dif:.3f}")
    print(f"Final Registered Run ID: {run.info.run_id}")
    print("=" * 50)